# QureML — IBM Quantum Hardware Validation (Parkinson's, Control A)

Standalone, self-contained notebook. Does NOT depend on any other notebook being run first.

What this does:
1. Loads UCI Parkinson's dataset fresh, reproduces the exact same preprocessing/split as `02_ablation_sweep.ipynb` (seed=42).
2. Trains Control A (full hybrid, trainable quantum layer) on the simulator — a few seconds, standard.
3. Picks a small balanced subset (5 positive + 5 negative = 10 samples) from the held-out test set.
4. Runs ONLY the already-trained quantum circuit's forward pass on a real IBM backend for those 10 samples (inference only — no training on hardware, keeps this well inside the free 10 min/month budget).
5. Compares real-hardware predictions against the simulator's predictions on the same 10 samples, and saves a comparison table.

Before running: add your IBM Quantum API key as a Kaggle Secret named `IBM_QUANTUM_TOKEN` (Add-ons -> Secrets in the notebook editor) rather than hardcoding it, since this notebook may get saved/shared as a proof artifact.


In [ ]:
!pip install -q -U pennylane ucimlrepo qiskit-ibm-runtime


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from ucimlrepo import fetch_ucirepo

N_QUBITS = 6
N_LAYERS = 2
EPOCHS = 60
LR = 0.05
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
print("Config: N_QUBITS =", N_QUBITS, "| N_LAYERS =", N_LAYERS, "| EPOCHS =", EPOCHS, "| LR =", LR)


In [ ]:
# ---- Load Parkinson's dataset (UCI id=174), same as 02_ablation_sweep.ipynb ----
parkinsons = fetch_ucirepo(id=174)
X_raw = parkinsons.data.features.to_numpy()
y = parkinsons.data.targets["status"].to_numpy().astype(int)
print("Parkinson's dataset:", X_raw.shape, "| positive class rate:", y.mean().round(3))

# ---- Fixed 80/20 stratified split, seed=42 (matches the main sweep's fixed_test_split) ----
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=SEED, stratify=y
)
print("Train:", X_train_raw.shape, "| Test:", X_test_raw.shape)


In [ ]:
# ---- Preprocessing: StandardScaler -> PCA(6) -> MinMaxScaler(0, pi), fit on TRAIN only ----
scaler = StandardScaler().fit(X_train_raw)
X_train_std = scaler.transform(X_train_raw)
X_test_std = scaler.transform(X_test_raw)

pca = PCA(n_components=N_QUBITS, random_state=SEED).fit(X_train_std)
X_train_pca = pca.transform(X_train_std)
X_test_pca = pca.transform(X_test_std)

angle_scaler = MinMaxScaler(feature_range=(0, np.pi)).fit(X_train_pca)
X_train_enc = angle_scaler.transform(X_train_pca).astype(np.float32)
X_test_enc = angle_scaler.transform(X_test_pca).astype(np.float32)

print("Encoded train/test shapes:", X_train_enc.shape, X_test_enc.shape)
print("Explained variance (train PCA):", pca.explained_variance_ratio_.sum().round(4))


In [ ]:
# ---- Circuit + HybridQNN definition (matches HybridQNN in the project notebooks) ----
import pennylane as qml

def variational_layer(weights, wires, entangle: bool):
    for i, w in enumerate(wires):
        qml.RY(weights[i, 0], wires=w)
        qml.RZ(weights[i, 1], wires=w)
    if entangle and len(wires) > 1:
        for i in range(len(wires)):
            qml.CNOT(wires=[wires[i], wires[(i + 1) % len(wires)]])

dev = qml.device("default.qubit", wires=N_QUBITS)

@qml.qnode(dev, interface="torch", diff_method="backprop")
def circuit(inputs, weights):
    # qml.AngleEmbedding (not a manual RY loop indexing inputs[i]) -- required so
    # PennyLane's TorchLayer batching broadcasts over the batch axis correctly.
    # A manual "for i in range(N_QUBITS): qml.RY(inputs[i], wires=i)" loop indexes
    # the BATCH axis instead of the feature axis once inputs is 2D, which breaks
    # training with a "shape invalid for input of size N_QUBITS" error.
    qml.AngleEmbedding(inputs, wires=range(N_QUBITS))
    for l in range(N_LAYERS):
        variational_layer(weights[l], wires=range(N_QUBITS), entangle=True)
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

weight_shapes = {"weights": (N_LAYERS, N_QUBITS, 2)}

class HybridQNN(nn.Module):
    def __init__(self, n_features, n_qubits=N_QUBITS):
        super().__init__()
        self.pre = nn.Linear(n_features, n_qubits)
        self.q_layer = qml.qnn.TorchLayer(circuit, weight_shapes)
        self.post = nn.Linear(n_qubits, 1)

    def forward(self, x):
        x = torch.tanh(self.pre(x)) * (torch.pi / 2)
        x = self.q_layer(x)
        x = self.post(x)
        return torch.sigmoid(x).squeeze(-1)

print("HybridQNN (Control A) defined.")


In [ ]:
# ---- Train Control A on the simulator (fast, seconds) ----
import time

torch.manual_seed(SEED)
model_a = HybridQNN(n_features=X_train_enc.shape[1], n_qubits=N_QUBITS)

X_train_t = torch.tensor(X_train_enc, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)

opt = torch.optim.Adam(model_a.parameters(), lr=LR)
loss_fn = nn.BCELoss()

t0 = time.time()
for epoch in range(EPOCHS):
    opt.zero_grad()
    preds = model_a(X_train_t)
    loss = loss_fn(preds, y_train_t)
    loss.backward()
    opt.step()
print(f"Trained Control A in {time.time() - t0:.2f}s | final train loss: {loss.item():.4f}")

model_a.eval()


In [ ]:
# ---- Pick a small balanced test subset: 5 positive + 5 negative ----
pos_idx = np.where(y_test == 1)[0][:5]
neg_idx = np.where(y_test == 0)[0][:5]
subset_idx = np.concatenate([pos_idx, neg_idx])

X_subset = X_test_enc[subset_idx]
y_subset = y_test[subset_idx]
X_subset_t = torch.tensor(X_subset, dtype=torch.float32)

print("Subset size:", len(subset_idx), "| positives:", y_subset.sum(), "| negatives:", (y_subset == 0).sum())


In [ ]:
# ---- Simulator baseline predictions on this exact subset ----
with torch.no_grad():
    sim_probs = model_a(X_subset_t).numpy()
sim_preds = (sim_probs > 0.5).astype(int)

print("Simulator probs:", sim_probs.round(4))
print(f"Simulator accuracy on 10-sample subset: {(sim_preds == y_subset).mean():.2f}")


In [ ]:
# ---- Extract the classical pre-layer output (angle-encoded inputs) and the trained
#      quantum weights -- these are what get sent to real hardware ----
with torch.no_grad():
    pre_out = torch.tanh(model_a.pre(X_subset_t)) * (torch.pi / 2)
quantum_weights = model_a.q_layer.weights.detach().numpy()  # shape (N_LAYERS, N_QUBITS, 2)
print("pre_out shape:", pre_out.shape, "| quantum_weights shape:", quantum_weights.shape)


In [ ]:
# ---- Connect to IBM Quantum, pick least-busy real backend ----
# NOTE: IBM retired the old channel="ibm_quantum" auth path. The current unified
# IBM Quantum Platform uses channel="ibm_quantum_platform" (the default) and an
# `instance` CRN identifying which instance (e.g. your "open-instance") to use.
#
# Get the CRN from the IBM Quantum Platform dashboard -> Instances -> hover over
# your "open-instance" row -> click the copy icon next to its CRN. Save it as a
# second Kaggle Secret named IBM_INSTANCE_CRN (Add-ons -> Secrets), same way you
# added IBM_QUANTUM_TOKEN.
from kaggle_secrets import UserSecretsClient
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit import QuantumCircuit, transpile

secrets = UserSecretsClient()
IBM_TOKEN = secrets.get_secret("IBM_QUANTUM_TOKEN")
IBM_INSTANCE_CRN = secrets.get_secret("IBM_INSTANCE_CRN")

service = QiskitRuntimeService(
    channel="ibm_quantum_platform",
    token=IBM_TOKEN,
    instance=IBM_INSTANCE_CRN,
)
backend = service.least_busy(operational=True, simulator=False, min_num_qubits=N_QUBITS)
print("Selected backend:", backend.name)


In [ ]:
# ---- Build the same circuit for each of the 10 samples -- NO measurement gates,
#      since EstimatorV2 computes expectation values of given observables directly
#      rather than sampling bitstrings (this is the correct primitive for this job:
#      the simulator circuit returns qml.expval(PauliZ) per qubit, and EstimatorV2
#      is IBM's primitive built specifically for expectation-value estimation). ----
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

def build_circuit_no_measure(angles, weights):
    qc = QuantumCircuit(N_QUBITS)
    for i in range(N_QUBITS):
        qc.ry(float(angles[i]), i)
    for l in range(N_LAYERS):
        for i in range(N_QUBITS):
            qc.ry(float(weights[l, i, 0]), i)
            qc.rz(float(weights[l, i, 1]), i)
        for i in range(N_QUBITS):
            qc.cx(i, (i + 1) % N_QUBITS)
    return qc

def z_observables(n_qubits):
    # One single-qubit PauliZ observable per qubit, matching
    # [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)] on the simulator side.
    return [SparsePauliOp.from_sparse_list([("Z", [q], 1.0)], num_qubits=n_qubits) for q in range(n_qubits)]

pre_out_np = pre_out.numpy()
raw_circuits = [build_circuit_no_measure(pre_out_np[i], quantum_weights) for i in range(len(X_subset))]

pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
isa_circuits = [pm.run(qc) for qc in raw_circuits]
base_observables = z_observables(N_QUBITS)

isa_pubs = []
for qc in isa_circuits:
    mapped_obs = [obs.apply_layout(qc.layout) for obs in base_observables]
    isa_pubs.append((qc, mapped_obs))

print(f"Built and transpiled {len(isa_circuits)} circuits (ISA-mapped for {backend.name})")

# Diagnostic: how much did routing/SWAP overhead inflate the circuit? This tells
# us whether hardware connectivity mismatch is the likely noise source.
logical_2q_count = N_LAYERS * N_QUBITS  # one CNOT per qubit per layer in the ring
physical_2q_counts = [qc.count_ops().get("cx", 0) + qc.count_ops().get("ecr", 0) for qc in isa_circuits]
print(f"Logical two-qubit gate count (ideal ring, no routing): {logical_2q_count}")
print(f"Physical two-qubit gate count after routing (per circuit): {physical_2q_counts}")
print(f"Physical circuit depth (per circuit): {[qc.depth() for qc in isa_circuits]}")


In [ ]:
# ---- Submit via EstimatorV2 with resilience_level=2: readout-error mitigation
#      (TREX) + gate (Pauli) twirling + zero-noise extrapolation (ZNE), IBM's
#      standard professional-grade error mitigation stack. ZNE samples at
#      multiple noise factors, so this costs ~3x the runtime of an unmitigated
#      run -- still small for 10 shallow 6-qubit circuits, well inside budget. ----
from qiskit_ibm_runtime import EstimatorV2 as Estimator

estimator = Estimator(mode=backend)
estimator.options.resilience_level = 2
estimator.options.default_shots = 1024
# Dynamical decoupling is NOT covered by resilience_level (that's TREX + twirling
# + ZNE only) -- it's a separate technique that inserts pulse sequences during
# idle qubit time to combat dephasing. XY4 is the more robust sequence choice.
estimator.options.dynamical_decoupling.enable = True
estimator.options.dynamical_decoupling.sequence_type = "XY4"
estimator.options.dynamical_decoupling.scheduling_method = "asap"

job = estimator.run(isa_pubs)
print("Job ID:", job.job_id())
print("Waiting for hardware result (resilience_level=2, ~3x runtime for ZNE)...")
result = job.result()
print("Job finished.")


In [ ]:
# ---- Read mitigated expectation values -> classical post-layer (same as sim) ----
hw_probs = []
for pub_result in result:
    expvals = np.array(pub_result.data.evs)  # shape (N_QUBITS,), ZNE/TREX-mitigated
    with torch.no_grad():
        post_out = model_a.post(torch.tensor(expvals, dtype=torch.float32).unsqueeze(0))
        prob = torch.sigmoid(post_out).item()
    hw_probs.append(prob)

hw_probs = np.array(hw_probs)
hw_preds = (hw_probs > 0.5).astype(int)

print("Hardware (mitigated) probs:", hw_probs.round(4))


In [ ]:
# ---- Compare hardware vs simulator, save results ----
import os
os.makedirs("results", exist_ok=True)

acc_hw = (hw_preds == y_subset).mean()
acc_sim = (sim_preds == y_subset).mean()
agreement = (hw_preds == sim_preds).mean()
mean_abs_diff = np.abs(hw_probs - sim_probs).mean()

print(f"Backend used: {backend.name}")
print(f"Simulator accuracy on subset:      {acc_sim:.2f}")
print(f"Real-hardware accuracy on subset:  {acc_hw:.2f}")
print(f"Prediction agreement (hw vs sim):  {agreement:.2f}")
print(f"Mean |hw_prob - sim_prob|:         {mean_abs_diff:.4f}")

comparison_df = pd.DataFrame({
    "true_label": y_subset,
    "sim_prob": sim_probs.round(4),
    "hw_prob": hw_probs.round(4),
    "sim_pred": sim_preds,
    "hw_pred": hw_preds,
    "agree": sim_preds == hw_preds,
})
comparison_df.to_csv("results/ibm_hardware_validation.csv", index=False)
comparison_df
